# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. It walks you through:
- Loading data from the Croissant schema
- Listing available record sets and fields by their `@id`
- Loading, transforming, and visualizing records

### Dataset Source
The dataset source is provided via this Croissant schema URL:

<https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json>

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview

List all available record sets, their `@id`s, and associated fields and columns by `@id`.

Inspect the dataset schema to determine key components for loading records.

In [ ]:
# Utility to pretty print record sets, fields, and columns (by @id and label/name)

print("Available Record Sets (by @id):\n")
for rs in dataset.metadata.record_sets:
    print(f"  RecordSet @id: {rs.id}, label: {getattr(rs, 'name', '')}")

    print("    Fields:")
    for field in rs.fields:
        print(f"      Field @id: {field.id}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")
        # If the field has columns (for tabular data)
        if hasattr(field, 'columns') and field.columns:
            print("        Columns:")
            for col in field.columns:
                print(f"          Column @id: {col.id}, header: {getattr(col, 'name', '')}")
    print()

Let's preview some sample records from a record set by its `@id`.

**Choose a `@id` from the record set list above for subsequent steps.**

In [ ]:
# Example: Preview records for the main protein table record set
# Replace the @id below with an actual RecordSet @id from the above output--
# In this dataset, there is likely a main record set with a descriptive id, e.g., 'protein_table',
# but you must confirm from the previous cell which id(s) are listed.

# Example value, adjust if needed:
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/dc2b7e0b-4874-4847-aa66-3b8e7406cd1d'  # <-- Replace with correct RecordSet @id

print(f"\nFirst 3 records from RecordSet: {main_record_set_id}")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(json.dumps(record, indent=2))
    if i >= 2:
        break

## 3. Data Extraction

Load all available records from one or more record sets into pandas DataFrames.

- The `records` method yields rows as dicts by field `@id`.
- You can change which record sets are loaded by changing their `@id` in `record_sets`.

In [ ]:
# List of all available record set @ids to extract (use those printed above)
record_sets = [
    main_record_set_id,  # Add more if needed
]

dataframes = {}
for rs_id in record_sets:
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for {rs_id}, shape: {df.shape}")
    print(f"Columns (@id): {df.columns.tolist()}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps:
- Filter records based on values of specific columns (by their field `@id`)
- Normalize a numeric column
- Optionally group by a categorical field (by `@id`)

Refer to the columns listed above to select the right `@id`s below.

In [ ]:
# Example EDA on the main DataFrame
# Update these @ids to match the field/column IDs from your dataset

record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Select a numeric field (@id) for analysis, e.g., coverage percentage, peptide counts, or abundance
numeric_field_id = 'https://api.app.sen.science/frontiers/7154140/3d86ae9f-dbff-4d57-9242-fde241e482b9'  # <-- e.g., "cr:field/coverage_percentage" (@id)

threshold = 10  # Example: filter coverage > 10%
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Optionally group by another field (@id), e.g., by protein type or gene
group_field_id = 'https://api.app.sen.science/frontiers/7154140/952b20b7-23f2-449c-b4c5-e2e84671a53e'  # <-- e.g., "cr:field/gene_symbol" (@id)

if group_field_id in df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped.head())


## 5. Visualization

Let's visualize the distribution of a numeric field or compare groups.

We'll use matplotlib to plot a histogram of the normalized numeric field and a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of normalized numeric field
plt.figure(figsize=(8,4))
filtered_df[norm_col].hist(bins=30)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(norm_col)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if group_field is present and non-null)
if group_field_id in filtered_df.columns:
    grouped = filtered_df.dropna(subset=[group_field_id])
    top_groups = grouped[group_field_id].value_counts().nlargest(5).index
    group_subset = grouped[grouped[group_field_id].isin(top_groups)]
    plt.figure(figsize=(10,6))
    group_subset.boxplot(column=norm_col, by=group_field_id)
    plt.title(f"Boxplot of normalized {numeric_field_id} by {group_field_id} (top 5 groups)")
    plt.suptitle('')
    plt.xlabel(group_field_id)
    plt.ylabel(norm_col)
    plt.show()

## 6. Conclusion

- We successfully loaded and explored the FAIR² human mass spectrometry dataset using only Croissant metadata and `mlcroissant`.
- The record set and field `@id`s enable robust and schema-driven analyses.
- Data pre-processing and visualization steps reveal value distributions and group-level patterns, readying for more domain-specific work.

For further analysis, consult the schema documentation to discover additional fields or relationships, or reference the [mlcroissant documentation](https://github.com/mlcommons/croissant/blob/main/docs/using-python.md) for more advanced extraction and transformation ideas.